# 1-A: Workflow, where the program decides the route

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AluminumShark/langgraph-skills-workshop/blob/main/en/notebooks/1-A_workflow.ipynb)

> 📍 **Matching slides**: Session 1, "Stop 2: Workflow 1-A", slides **16–28** (class time 0:35–1:00)

## What this notebook does

This notebook builds the first version of the Homework Helper, and its **route is decided by the program** (this approach is called a workflow):
it checks the syntax first. If the syntax is wrong, it explains the error and ends, **skipping the format check**. Only when the syntax passes does it generate feedback and then check the format; if the format fails and the limit has not been reached, it generates again.
What each step does and where the next step goes were all written in code beforehand. The AI only handles one of the steps.

We write it with **LangGraph**, a Python package for **connecting a few functions with arrows so that one piece of data flows through them in order**.
This session introduces three core components and compares them with the Python you already know. Think of a bubble-tea shop:

| Python you know | LangGraph name | Bubble-tea shop | In the helper |
| --- | --- | --- | --- |
| dict | **State** | The order slip that travels with the drink | One slip records the homework, whether the syntax is right, the feedback, and how many times it has been generated |
| function | **Node** | One workstation | `check_syntax`, `give_feedback`... each does just one thing |
| calling in order | **Edge** | The arrows between workstations | After the feedback is written, check the format |
| if / else | **Conditional edge** | A hot drink skips the ice station | If the syntax is wrong, go to "explain the error" instead |
| while loop | An edge that points back | The sweetness is wrong, so send it back to be remade | The format fails and the limit is not reached, so generate again |

| Part | What you will do | Matching slides |
| --- | --- | --- |
| Part 1 | Write out `graph.py`, draw the flow diagram, and find every Node and Edge | 17–23 |
| Part 2 | No AI yet: switch between three homework samples and watch the order slip fill in cell by cell | 24–25 |
| Part 3 | Put the AI into one node; adjust "Maximum generations" and "Feedback character limit" | 26–28 |
| Optional after class | The message-merging rule | 46 |

## How to use it

1. First run **"0. Environment setup"**.
2. Every part: **🤔 write your prediction first → ▶ run → 👀 compare**. The only things you change in this notebook are form fields: **homework, maximum generations, feedback character limit**.
3. **In class (required)**: Parts 1–3. **Optional after class**: the message-merging rule.
4. When you see a long program with `%%writefile`, just run it to write the file; in this session read the output first, and you don't need to understand every line.
5. The parts need the `graph.py` written in Part 1. If the module can't be found, run the environment setup and the Part 1 file-writing cell first, then come back to the current part.

## 🎯 The mini project for these two sessions: the Homework Helper

The **Homework Helper** is a small program that helps check Python homework. You give it a piece of homework, and it replies with the review result.
In the workflow version (1-A, workflow), a syntax error gets an error reply first; only code with valid syntax goes on to the AI, which writes feedback containing **"Strengths:" and "Improvements:"**.
If the format check keeps failing, the flow stops when it reaches the maximum number of generations and marks the result as not passed. "Stopped" must never be read as "passed".

**Keep three things apart:** valid syntax (Python can read it) ≠ runs successfully (no error while running) ≠ meets the assignment (matches what the assignment asks for).

We chose it because it happens to use all three kinds of things (and you can also combine them):

| What the helper needs | Example | What we use |
| --- | --- | --- |
| Steps controlled by a flow | Check syntax first; report errors; only on a pass, write feedback and check its format | A **workflow** defined in code (LangGraph, Session 1) |
| Things that need judgment | How to word the feedback, whether to actually run the code and see | Leave it to the **AI** (agent, Session 1) |
| Ways of working and rules | Should the feedback be gentle or strict, what a good variable name looks like | A **handbook** written for the AI to read (Skill, Session 2). Rules that code can check (such as variable naming) can also come with a script or go into the workflow; 2-B compares them |

👉 **This notebook covers:** Use LangGraph to connect the helper's four workstations (check syntax, explain error, write feedback, check format) into one fixed flow. First see how the data flows without any AI, then put the AI into the "write feedback" station.

<details><summary>Open this when you need to look up a component name (no need to memorize it)</summary>

| Component | What it does, in plain words |
| --- | --- |
| `check_syntax` (check syntax) | Only checks whether the code's "grammar" is right, for example whether brackets are closed or a colon is missing. It **does not run the code** |
| `explain_error` (explain the error) | When the syntax is wrong, tells the student in one sentence which line is wrong |
| `give_feedback` (write feedback) | Once a model is connected in 1-A, asks the AI to write "Strengths + Improvements" feedback; the next station checks whether it matches the format |
| `check_format` (check format) | Checks the order of the two markers, that each part is non-empty, and the total character count. It does not verify that the feedback content is correct |
| Tools: `run_code`, `check_indent`, `read_spec` (from 1-B on); `check_naming` (from 2-B on) | Run the homework under teaching limits, check indentation, read the assignment requirements, check variable names |
| Handbooks: `friendly-feedback`, `strict-review`; `naming-check` (2-B), `detailed-feedback` (the 2-C security experiment) | Tell the AI whether feedback should be gentle and encouraging or strict; the naming-check handbook comes with a script; the last one hides a suspicious instruction so you can practice checking an outside handbook |

</details>

**Reminder:** run "0. Setup" first, then follow each part: setup cell → predict → run → compare.
`%%writefile` means "save this cell's content as a file"; edits only take effect after you run the cell. Re-running a setup cell may overwrite the handbooks, so save your own edits first.

In [ ]:
#@title 0. Setup (run it the first time you open this notebook, or after the runtime is reset)
#@markdown **Model source**: `colab` = the model built into Colab (no API key needed; availability and quota depend on your account); `gemini` = the instructor's Gemini API key (the backup option)
model_source = "colab"  #@param ["colab", "gemini"]
#@markdown If you pick `gemini`: we recommend adding `GOOGLE_API_KEY` under 🔑 "Secrets" in the left sidebar. You can also paste the key below (⚠️ if you paste it, do not share this notebook)
INSTRUCTOR_API_KEY = ""  #@param {type:"string"}
#@markdown Advanced (you usually don't need to change these): set a model name, or leave it empty to use the default
colab_model_name = ""  #@param {type:"string"}
gemini_model_name = "gemini-flash-latest"  #@param {type:"string"}

import os, shutil, subprocess, sys
from pathlib import Path

# 1. Install packages with uv (limited to compatible version ranges; time depends on your network)
if not os.environ.get("WORKSHOP_SKIP_INSTALL"):
    print("⏳ Installing LangGraph / LangChain with uv ...")
    if shutil.which("uv") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(f'uv pip install --python "{sys.executable}" -q "langgraph>=1.1,<2" "langchain>=1.2,<2" "langchain-google-genai>=4,<5" "langgraph-checkpoint-sqlite>=3,<4"', shell=True, check=True)

# 2. Write the shared files this notebook uses
RB = Path("/content/review-bot")
for d in ['tools', 'deploy', 'skills/friendly-feedback', 'skills/strict-review', 'skills/naming-check/scripts', 'skills/detailed-feedback/scripts']:
    (RB / d).mkdir(parents=True, exist_ok=True)
_files = {
    'llm.py': r'''"""Model adapter: wraps either "Colab's built-in model" or "the instructor's Gemini API key" into one model object.

Students don't need to read this file. Every later cell only writes model.invoke(...) and model.bind_tools(...),
and it works the same no matter which provider is behind it.

- colab : google.colab.ai.generate_text (works with a free account, no API key needed)
- gemini: langchain_google_genai.ChatGoogleGenerativeAI (needs GOOGLE_API_KEY)

generate_text takes only one piece of text and returns only one piece of text. It has no native tool calling.
So ColabAIChat adds a small adapter:
1. It joins the messages into one text (system instructions, user, AI, and tool results each get a header)
2. When tools are bound, it appends the tool list and asks the model to "reply with a single line of JSON if you need a tool"
3. It parses the model's JSON into tool_calls, in exactly the same format as native tool calling
This lets you see that the model only ever "says" which tool it wants.
"""

from __future__ import annotations

import json
import os
import re
import time
import uuid
from pathlib import Path
from typing import Any

from langchain_core.language_models.chat_models import BaseChatModel
from langchain_core.messages import AIMessage, BaseMessage, ToolMessage
from langchain_core.outputs import ChatGeneration, ChatResult
from langchain_core.utils.function_calling import convert_to_openai_tool

SETTINGS_FILE = Path(__file__).with_name(".llm.json")
DEFAULT_GEMINI_MODEL = "gemini-flash-latest"


def estimate_tokens(text: str) -> int:
    """Rough token estimate: about 4 characters per token for English text.
    This is only a guess. A real tokenizer can count noticeably more or fewer."""
    return max(1, round(len(text) / 4))


# ---------- Join the messages into one text ----------

def _describe_tools(tools: list[dict]) -> str:
    lines = []
    for t in tools:
        fn = t.get("function", t)
        props = fn.get("parameters", {}).get("properties", {})
        params = ", ".join(f"{k}: {v.get('type', 'string')}" for k, v in props.items())
        desc = " ".join((fn.get("description") or "").split())
        lines.append(f"- {fn['name']}({params}): {desc}")
    return "\n".join(lines)


def _messages_to_text(messages: list[BaseMessage], tools: list[dict] | None, tool_choice: Any) -> str:
    parts = []
    for m in messages:
        if m.type == "system":
            parts.append(f"=== System instructions ===\n{m.text}")
        elif m.type == "human":
            parts.append(f"=== User ===\n{m.text}")
        elif m.type == "ai":
            if m.tool_calls:
                calls = [{"name": c["name"], "args": c["args"]} for c in m.tool_calls]
                parts.append("=== AI ===\n" + json.dumps({"tool_calls": calls}, ensure_ascii=False))
            else:
                parts.append(f"=== AI ===\n{m.text}")
        elif m.type == "tool":
            name = getattr(m, "name", None) or "tool"
            parts.append(f"=== Result of tool {name} ===\n{m.text}")
    if tools:
        must = ""
        if tool_choice in ("any", "required"):
            must = "This time you must call at least one tool.\n"
        elif isinstance(tool_choice, str) and tool_choice not in ("auto", "none"):
            must = f"This time you must call {tool_choice}.\n"
        parts.append(
            "=== Tools you can use ===\n"
            f"{_describe_tools(tools)}\n\n"
            "=== Reply rules ===\n"
            f"{must}"
            "If you need to use a tool, reply with a single line of JSON only. Add no other text and no ``` marks:\n"
            '{"tool_calls": [{"name": "tool name", "args": {"parameter name": "parameter value"}}]}\n'
            "If you don't need a tool (for example, you already have enough information), just answer in plain text and output no JSON."
        )
    parts.append("=== AI ===")
    return "\n\n".join(parts)


def _parse_reply(text: str, tools: list[dict] | None) -> AIMessage:
    """If the model replied with a JSON note, turn it into tool_calls. Otherwise treat it as plain text."""
    meta = {"raw_text": text, "model_provider": "colab"}
    if tools:
        s = text.strip()
        s = re.sub(r"^```(?:json)?\s*|\s*```$", "", s)
        start, end = s.find("{"), s.rfind("}")
        if start != -1 and end > start and ("tool_calls" in s or s.startswith("{")):
            try:
                data = json.loads(s[start : end + 1])
            except json.JSONDecodeError:
                data = None
            calls = None
            if isinstance(data, dict):
                if isinstance(data.get("tool_calls"), list):
                    calls = data["tool_calls"]
                elif "name" in data or "tool" in data:
                    calls = [data]
            if calls:
                tool_calls = []
                for c in calls:
                    if not isinstance(c, dict):
                        continue
                    name = c.get("name") or c.get("tool")
                    args = c.get("args") or c.get("arguments") or {}
                    if isinstance(args, str):
                        try:
                            args = json.loads(args)
                        except json.JSONDecodeError:
                            args = {}
                    if name:
                        tool_calls.append(
                            {"name": name, "args": args, "id": f"call_{uuid.uuid4().hex[:8]}", "type": "tool_call"}
                        )
                if tool_calls:
                    return AIMessage(content="", tool_calls=tool_calls, response_metadata=meta)
    return AIMessage(content=text, response_metadata=meta)


class ColabAIChat(BaseChatModel):
    """Wraps google.colab.ai.generate_text as a LangChain chat model."""

    model_name: str | None = None
    max_retries: int = 3

    @property
    def _llm_type(self) -> str:
        return "colab-ai"

    def bind_tools(self, tools, *, tool_choice=None, **kwargs):
        formatted = [convert_to_openai_tool(t) for t in tools]
        return self.bind(tools=formatted, tool_choice=tool_choice)

    def _generate(self, messages, stop=None, run_manager=None, **kwargs) -> ChatResult:
        from google.colab import ai

        tools = kwargs.get("tools")
        prompt = _messages_to_text(messages, tools, kwargs.get("tool_choice"))
        extra = {"model_name": self.model_name} if self.model_name else {}
        for attempt in range(self.max_retries):
            try:
                text = ai.generate_text(prompt, **extra) or ""
                break
            except Exception:
                if attempt == self.max_retries - 1:
                    raise
                time.sleep(2 * (attempt + 1))
        message = _parse_reply(text, tools)
        message.usage_metadata = {
            "input_tokens": estimate_tokens(prompt),
            "output_tokens": estimate_tokens(text),
            "total_tokens": estimate_tokens(prompt) + estimate_tokens(text),
        }
        message.response_metadata["token_count"] = "estimate (Colab's built-in model does not report usage)"
        return ChatResult(generations=[ChatGeneration(message=message)])


# ---------- Public: settings and getting the model ----------

def load_settings() -> dict:
    try:
        return json.loads(SETTINGS_FILE.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_settings(provider: str, colab_model: str = "", gemini_model: str = "") -> None:
    """Remember which model provider was chosen (the API key is not saved)."""
    SETTINGS_FILE.write_text(
        json.dumps(
            {"provider": provider, "colab_model": colab_model, "gemini_model": gemini_model or DEFAULT_GEMINI_MODEL},
            ensure_ascii=False,
        ),
        encoding="utf-8",
    )


def _find_api_key() -> str | None:
    if os.environ.get("GOOGLE_API_KEY"):
        return os.environ["GOOGLE_API_KEY"]
    try:
        from google.colab import userdata

        key = userdata.get("GOOGLE_API_KEY")
        if key:
            os.environ["GOOGLE_API_KEY"] = key
            return key
    except Exception:
        pass
    return None


def get_model() -> BaseChatModel:
    cfg = load_settings()
    # In the cloud there is no settings file: use the environment variable WORKSHOP_PROVIDER=gemini (the key is read from GOOGLE_API_KEY)
    provider = os.environ.get("WORKSHOP_PROVIDER") or cfg.get("provider", "colab")
    if provider == "gemini":
        key = _find_api_key()
        if not key:
            raise RuntimeError(
                "You chose gemini but no API key was found. Add GOOGLE_API_KEY under 🔑 Secrets on the left, "
                "or paste the key from the instructor into the form in \"0. Environment setup\", then rerun that cell."
            )
        from langchain_google_genai import ChatGoogleGenerativeAI

        return ChatGoogleGenerativeAI(
            model=cfg.get("gemini_model") or DEFAULT_GEMINI_MODEL, google_api_key=key, max_retries=3
        )
    try:
        from google.colab import ai  # noqa: F401
    except ImportError as exc:
        raise RuntimeError("google.colab.ai only works on Google Colab. If you are not on Colab, choose gemini instead.") from exc
    return ColabAIChat(model_name=cfg.get("colab_model") or None)
''',
    'workshop.py': r'''"""Small helpers shared by the workshop notebooks (students don't need to read this).

- prepare()                     : the first cell of every part calls it to reconnect the model and reload edited files
- sample_homework               : the sample homework in the form dropdown
- record() / print_history()    : save the results that experiment cells name, then list them by title in the last cell
- show_graph()                  : draw a LangGraph as a diagram
- save_to_drive() / download_zip()
"""

from __future__ import annotations

import datetime as _dt
import importlib
import json
import shutil
import sys
from pathlib import Path

from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages import BaseMessage

from llm import get_model, estimate_tokens  # noqa: F401  (so notebooks can write: from workshop import estimate_tokens)

__all__ = [
    "BASE", "prepare", "sample_homework", "load_homework", "record", "print_state", "print_order_slip", "print_history", "clear_history",
    "ModelCallCounter", "show_graph", "save_to_drive", "download_zip", "estimate_tokens",
]

BASE = Path(__file__).resolve().parent
HISTORY_FILE = BASE / "history.jsonl"

sample_homework = {
    "Missing parenthesis": "print('hi'\n",
    "Correct program": "def add(a, b):\n    return a + b\n\nprint(add(2, 3))\n",
    "Empty": "",
    "Variable typo": "def add(a, b):\n    total = a + b\n    return summ\n\nprint(add(2, 3))\n",
    "Messy indentation": "def average(scores):\n  total = 0\n  for s in scores:\n        total += s\n  return total / len(scores)\n\nprint(average([90, 85, 77]))\n",
    "Inconsistent naming": (
        "def CalcAverage(scoreList):\n"
        "    Total = 0\n"
        "    for s in scoreList:\n"
        "        Total += s\n"
        "    avgScore = Total / len(scoreList)\n"
        "    return avgScore\n"
        "\n"
        "myScores = [90, 85, 77]\n"
        "MAX = 100\n"
        "result_2 = CalcAverage(myScores)\n"
        "print(result_2, '/', MAX)\n"
    ),
}


def load_homework(name: str) -> str:
    """When the form says "My homework", read /content/review-bot/my_homework.py. Otherwise take it from the samples."""
    if name.startswith("My homework"):
        path = BASE / "my_homework.py"
        if not path.exists():
            raise FileNotFoundError('There is no "My homework" yet. Run the cell above that has %%writefile ... my_homework.py first.')
        return path.read_text(encoding="utf-8")
    return sample_homework[name]


def prepare():
    """The start of every part: put review-bot on the import path, reload edited files, and return the model."""
    if str(BASE) not in sys.path:
        sys.path.insert(0, str(BASE))
    importlib.invalidate_caches()
    for name, mod in list(sys.modules.items()):
        path = getattr(mod, "__file__", None) or ""
        if path and Path(path).resolve().is_relative_to(BASE) and name not in ("workshop", "llm"):
            try:
                importlib.reload(mod)
            except Exception:
                sys.modules.pop(name, None)
    _make_mcp_stdio_work_in_notebook()
    return get_model()


def _make_mcp_stdio_work_in_notebook() -> None:
    """mcp's stdio_client connects the MCP server's error output to sys.stderr.
    On Colab (Linux), sys.stderr has a real file descriptor, so this works. Jupyter on Windows has none, so we redirect to a log file."""
    try:
        sys.stderr.fileno()
        return
    except Exception:
        pass
    try:
        import mcp.client.stdio as mcp_stdio
    except ImportError:
        return
    (BASE / "deploy").mkdir(exist_ok=True)
    log = open(BASE / "deploy" / "mcp_server.log", "a", encoding="utf-8")
    mcp_stdio.stdio_client.__wrapped__.__defaults__ = (log,)


# ---------- State history ----------

def _serializable(obj):
    if isinstance(obj, BaseMessage):
        d = {"type": obj.type, "content": obj.text}
        if getattr(obj, "tool_calls", None):
            d["tool_calls"] = [{"name": c["name"], "args": c["args"]} for c in obj.tool_calls]
        if obj.type == "tool":
            d["name"] = getattr(obj, "name", None)
        return d
    if isinstance(obj, dict):
        return {str(k): _serializable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_serializable(v) for v in obj]
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def record(title: str, content) -> None:
    """Save this result (usually a State) into history.jsonl."""
    row = {"time": _dt.datetime.now().strftime("%H:%M:%S"), "title": title, "content": _serializable(content)}
    with HISTORY_FILE.open("a", encoding="utf-8") as f:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


def _short(text, n=300):
    text = str(text)
    return text if len(text) <= n else text[:n] + f"... ({len(text) - n} more characters)"


def print_state(state: dict, indent: str = "  ") -> None:
    for k, v in state.items():
        if k == "messages":
            print(f"{indent}{k}: ({len(v)} messages)")
            for m in v:
                m = _serializable(m)
                if isinstance(m, dict) and m.get("tool_calls"):
                    print(f"{indent}  [{m['type']}] calls {m['tool_calls']}")
                elif isinstance(m, dict):
                    print(f"{indent}  [{m.get('type')}] {_short(m.get('content', ''), 200)}")
        else:
            print(f"{indent}{k}: {_short(repr(v) if not isinstance(v, str) else repr(v), 300)}")


def print_order_slip(step_no: int, state: dict, prev_state: dict | None = None, skip_keys: tuple = ("code",)) -> None:
    """stream_mode="values" gives the whole order slip at every step. This also marks which fields this step newly wrote."""
    if not prev_state:
        note = "start: blank order slip"
    else:
        changed_keys = [k for k in state if state.get(k) != prev_state.get(k)]
        note = "newly written: " + (", ".join(changed_keys) if changed_keys else "(nothing changed)")
    print(f"Step {step_no} ({note})")
    print("  ", {k: (_short(v, 120) if isinstance(v, str) else v) for k, v in state.items() if k not in skip_keys})


def print_history() -> None:
    if not HISTORY_FILE.exists():
        print("There are no records yet. Run a few parts above first, then come back to this cell.")
        return
    rows = [json.loads(line) for line in HISTORY_FILE.read_text(encoding="utf-8").splitlines() if line.strip()]
    print(f"📸 {len(rows)} records (saved in {HISTORY_FILE})\n")
    for i, row in enumerate(rows, 1):
        print(f"===== {i}. [{row['time']}] {row['title']} =====")
        content = row["content"]
        if isinstance(content, dict):
            print_state(content)
        elif isinstance(content, list):
            for j, item in enumerate(content, 1):
                print(f"  --- step {j} ---")
                if isinstance(item, dict):
                    print_state(item, "    ")
                else:
                    print(f"    {_short(item)}")
        else:
            print(f"  {_short(content, 1000)}")
        print()


def clear_history() -> None:
    HISTORY_FILE.unlink(missing_ok=True)


# ---------- Count how many times the model was called ----------

class ModelCallCounter(BaseCallbackHandler):
    """Put it in config={"callbacks": [counter]}. It adds 1 every time the model is called."""

    def __init__(self):
        self.count = 0

    def on_chat_model_start(self, serialized, messages, **kwargs):
        self.count += 1

    def on_llm_start(self, serialized, prompts, **kwargs):
        self.count += 1


# ---------- Draw the graph ----------

def show_graph(graph) -> None:
    g = graph.get_graph()
    try:
        from IPython.display import Image, display

        display(Image(g.draw_mermaid_png()))
    except Exception:
        print("(The drawing service is unreachable for now, so here is a text version. You can paste it into https://mermaid.live to see the diagram.)\n")
        print(g.draw_mermaid())


# ---------- Saving ----------

def save_to_drive() -> None:
    from google.colab import drive

    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/langgraph-skills-workshop") / _dt.datetime.now().strftime("%Y%m%d-%H%M%S")
    shutil.copytree(BASE, destination, ignore=shutil.ignore_patterns("__pycache__", ".llm.json"))
    print(f"✅ Saved to Google Drive: {destination}")
    print('   (Save the notebook itself with the menu above: "File → Save a copy in Drive")')


def download_zip() -> None:
    from google.colab import files

    name = BASE.parent / ("review-bot-" + _dt.datetime.now().strftime("%Y%m%d-%H%M%S"))
    zip_path = shutil.make_archive(str(name), "zip", BASE.parent, BASE.name)
    print(f"✅ Packed: {zip_path}")
    files.download(zip_path)
''',
}
for rel, content in _files.items():
    (RB / rel).write_text(content, encoding="utf-8")
sys.path.insert(0, str(RB))
# Make Python look again for the files we just wrote (if another cell ran first, Python remembers "this folder does not exist")
import importlib
importlib.invalidate_caches()

# 3. Remember the model source (the API key is not saved) and test the connection
import llm
llm.save_settings(model_source, colab_model_name.strip(), gemini_model_name.strip())
if INSTRUCTOR_API_KEY.strip():
    os.environ["GOOGLE_API_KEY"] = INSTRUCTOR_API_KEY.strip()
from workshop import *
model = prepare()

reply = model.invoke("Say hello in one sentence to a classmate who just learned Python").text
print(f"✅ Setup complete. Model source: {model_source}")
print(f"🤖 The model says: {reply.strip()}")

### ✏️ (Optional) Bring in your own homework

The take-home assignment asks you to run **a piece of homework you wrote yourself**. Replace the code in the cell below with your homework and run it.
After that, choose **My homework** in the form's "homework" field and the helper will use it. You can skip this cell in class.
`run_code` does not allow `input()` or `open()`, and only a few standard modules can be imported. If your homework uses `input()`, change it to a fixed value first (for example `n = 5`).

In [ ]:
%%writefile /content/review-bot/my_homework.py
# Replace this with a piece of Python homework you wrote yourself, then remember to run this cell
def average(scores):
    total = 0
    for s in scores:
        total = total + s
    return total / len(scores)

print(average([90, 85, 77]))


---
## Part 1: LangGraph just connects functions with arrows

> 📍 **Matching slides**: slide 17 "LangGraph: connecting functions with arrows", 18 "The Homework Helper's flow: four workstations", 20 "Component 1: State, the order slip that travels along", 21 "Component 2: Node, one workstation", 22 "Component 3: Edge, the arrows between workstations", 23 "One table for the three components"

First, look at the whole flow diagram: what the helper's four workstations look like in code, and how they are connected.

The next cell saves the helper's program as `/content/review-bot/graph.py` (no changes needed, just run it).
If you want to read the program, there are only three kinds of things to look for:

- **State**: the `class State(TypedDict)` part, which is which cells the order slip has (see the table below).
- **Node**: functions like `def check_syntax(state)`. It receives the whole order slip and **returns only the cells it is responsible for**; LangGraph writes them back onto the slip for you.
- **Edge**: `add_edge` at the bottom (after A, always go to B) and `add_conditional_edges` (decide the next station depending on the situation, which is an if).

| Cell on the order slip | What it records |
| --- | --- |
| `code` | The homework the student turned in |
| `syntax_ok`, `error` | Whether the syntax is right, and which line is wrong (before checking, `syntax_ok` is `None` and `error` is an empty string) |
| `feedback` | The feedback for the student |
| `format_ok`, `format_problem` | Whether the feedback format passes, and the reason if it doesn't |
| `retries` | How many times the feedback has been written |

> What is "syntax"? It is Python's grammar rules, like parentheses must come in pairs and the `def` line must end with a colon.
> `check_syntax` only checks grammar and **does not run** the program. `print(unknown_name)` has correct syntax, but it fails when run because the name is not defined.
> Valid syntax, running successfully and meeting the assignment are three different checks. The helper in 1-A only checks syntax; `run_code`, which actually runs the program (you tried it in 1-0 Part 4), is not handed to the helper until 1-B.

### 🤔 Prediction questions
1. How many Nodes (workstations) will the flow diagram have?
2. Which arrows are "conditional edges" (the ones that fork)?
3. Is there an arrow that points backward? Which Python syntax does it resemble?

In [ ]:
%%writefile /content/review-bot/graph.py
"""1-A: the Homework Helper as a workflow. The route is decided by code.

START → check_syntax ─syntax OK→ give_feedback → check_format ─format OK or limit reached→ END
                     └syntax error→ explain_error → END       └failed and retries below the limit→ send back to give_feedback

Three components:
- State: one order slip (a dict) that travels from start to finish
- Node : one workstation (a function). It takes the State and returns only the fields it changed
- Edge : the arrows between workstations. A conditional edge is just an if that returns the name of the next station
"""

import ast
import re
from typing import TypedDict

from langgraph.graph import END, START, StateGraph


class State(TypedDict, total=False):
    code: str              # the homework the student submitted
    syntax_ok: bool | None # is the syntax valid? (None before it is checked)
    error: str             # where the syntax is wrong
    feedback: str          # the feedback for the student
    format_ok: bool | None # does the feedback pass the format check?
    format_problem: str    # why it failed (told to the AI when the feedback is sent back)
    retries: int           # how many times feedback has been generated, including the first


# ---------- Node 1: check_syntax (pure Python): only checks the grammar, never runs the code ----------

_ERROR_TRANSLATIONS = [
    (r"'(.)' was never closed", lambda m: f"missing a closing bracket {dict(zip('([{', ')]}')).get(m.group(1), '')}"),
    (r"unterminated string literal", lambda m: "the string is missing its closing quote"),
    (r"expected ':'", lambda m: "missing a colon :"),
    (r"unexpected indent", lambda m: "extra indentation"),
    (r"expected an indented block", lambda m: "this line needs to be indented"),
    (r"unindent does not match", lambda m: "the indentation doesn't line up"),
    (r"unmatched '(.)'", lambda m: f"an extra {m.group(1)}"),
    (r"invalid character", lambda m: "there is a full-width or unrecognized character"),
    (r"Perhaps you forgot a comma", lambda m: "maybe a comma is missing"),
    (r"invalid syntax", lambda m: "invalid syntax"),
]


def translate_syntax_error(msg: str) -> str:
    for pattern, wording in _ERROR_TRANSLATIONS:
        m = re.search(pattern, msg)
        if m:
            return wording(m)
    return msg


def check_syntax(state: State) -> dict:
    code = state.get("code", "")
    if not code.strip():
        return {"syntax_ok": False, "error": "the homework is empty, there is no code"}
    try:
        ast.parse(code)
    except SyntaxError as e:
        return {"syntax_ok": False, "error": f"line {e.lineno}: {translate_syntax_error(e.msg)}"}
    return {"syntax_ok": True, "error": ""}


# ---------- Conditional edge 1: syntax OK → write feedback; syntax error → explain the error (just an if) ----------

def route_after_syntax(state: State) -> str:
    if state["syntax_ok"]:
        return "give_feedback"
    return "explain_error"


# ---------- Node 2: explain_error (pure Python): one sentence saying which line is wrong ----------

def explain_error(state: State) -> dict:
    return {"feedback": f"Syntax: {state['error']}\nImprovements: fix this line first, then run it again."}


# ---------- Node 3: give_feedback (two versions: fixed text / handed to the AI) ----------

def give_feedback_fixed_text(state: State) -> dict:
    """The version with no AI yet: it returns a fixed piece of text."""
    return {
        "feedback": "Strengths: this homework passes the syntax check.\nImprovements: (this version has no AI yet, so the run result and the logic are not checked)",
        "retries": state.get("retries", 0) + 1,
    }


def make_give_feedback(model, feedback_char_limit: int):
    """The version with AI: the AI is just one line of code living inside this node."""

    def give_feedback(state: State) -> dict:
        prompt = (
            "You are a Python homework helper. Give feedback in English on the homework below.\n"
            "Format rules (you must follow them):\n"
            '- Write only two paragraphs: the first starts with "Strengths:", the second starts with "Improvements:"\n'
            f'- The whole feedback (including "Strengths:", "Improvements:", and line breaks) must be at most {feedback_char_limit} characters\n\n'
            f"Homework:\n```python\n{state['code']}\n```"
        )
        if state.get("format_problem"):
            prompt += f"\n\nYour previous feedback was rejected because: {state['format_problem']}. Fix it and rewrite."
        text = model.invoke(prompt).text
        return {"feedback": text.strip(), "retries": state.get("retries", 0) + 1}

    return give_feedback


# ---------- Node 4: check_format (a pure-Python checkpoint): needs two paragraphs and must not be too long ----------

def make_check_format(feedback_char_limit: int):
    def check_format(state: State) -> dict:
        feedback = state.get("feedback", "")
        problems = []
        # Check the label order and that both paragraphs are non-empty; line breaks are accepted.
        parts = re.fullmatch(r"\s*Strengths:\s*(.+?)\s*Improvements:\s*(.+?)\s*", feedback, re.DOTALL)
        if not parts or not all(part.strip() for part in parts.groups()):
            problems.append('it needs two paragraphs in order: "Strengths: ..." and "Improvements: ..."')
        if len(feedback) > feedback_char_limit:
            problems.append(f"too long: {len(feedback)} characters (including labels and line breaks), limit {feedback_char_limit} characters")
        return {"format_ok": not problems, "format_problem": "; ".join(problems)}

    return check_format


# ---------- Conditional edge 2: format OK or generation limit reached → end; otherwise send back to rewrite ----------

def make_route_after_format(max_retries: int):
    def route_after_format(state: State) -> str:
        if state["format_ok"]:
            return END
        if state.get("retries", 0) >= max_retries:
            return END  # reached the feedback generation limit; stopping does not mean the format passed
        return "give_feedback"

    return route_after_format


# ---------- Connect the components ----------

def build_helper(model=None, max_retries: int = 3, feedback_char_limit: int = 450):
    """With model=None it uses fixed text. max_retries is a legacy parameter name; it actually counts generations, including the first."""
    give_feedback = make_give_feedback(model, feedback_char_limit) if model else give_feedback_fixed_text

    builder = StateGraph(State)
    builder.add_node("check_syntax", check_syntax)
    builder.add_node("explain_error", explain_error)
    builder.add_node("give_feedback", give_feedback)
    builder.add_node("check_format", make_check_format(feedback_char_limit))

    builder.add_edge(START, "check_syntax")
    builder.add_conditional_edges("check_syntax", route_after_syntax, ["give_feedback", "explain_error"])
    builder.add_edge("explain_error", END)
    builder.add_edge("give_feedback", "check_format")
    builder.add_conditional_edges("check_format", make_route_after_format(max_retries), ["give_feedback", END])
    return builder.compile()


def blank_order_slip(code: str) -> State:
    return {"code": code, "syntax_ok": None, "error": "", "feedback": "", "format_ok": None,
            "format_problem": "", "retries": 0}


In [ ]:
#@title ▶ Run: draw the Homework Helper's flow diagram
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import graph

helper = graph.build_helper()        # No model yet: give_feedback uses fixed text
show_graph(helper)

g = helper.get_graph()
print("Nodes (workstations):", [n for n in g.nodes if n not in ("__start__", "__end__")])
print("Edges (arrows):")
for e in g.edges:
    print(f"  {e.source} -> {e.target}{'   <- conditional edge (if)' if e.conditional else ''}")

### 👀 What did you see?

- 4 workstations: `check_syntax`, `explain_error`, `give_feedback`, `check_format` (`__start__` and `__end__` are the start and end points).
- There are two places with conditional edges: after `check_syntax` it splits into "syntax OK -> give_feedback" and "syntax wrong -> explain_error"; after `check_format` it splits into "OK or limit reached -> end" and "fails and limit not reached -> send back to give_feedback".
- The arrow from `check_format -> give_feedback` that **points backward** is just like a `while` loop: the sweetness is wrong, so send it back to be remade.
- 🏅 Bonus question: in the `graph.py` cell above, find which lines these two conditional edges are written on (hint: look for `add_conditional_edges`).

---
## Part 2: No AI yet, guess then run three homework samples

> 📍 **Matching slides**: slide 24 "No AI yet: three homework samples, guess then run", slide 25 "Data flow: watch the order slip fill in"

The last part showed the "shape" of the flow diagram. This part feeds in real homework and shows how the data flows from station to station.

The nodes in this part are ordinary Python with no AI yet, and `give_feedback` returns a fixed piece of text.
The form below has a "homework" dropdown. **Pick "Missing parenthesis", "Correct program" and "Empty" in turn and run once for each.**

The program runs with `stream_mode="values"`, which means **after every station finishes, the whole order slip is printed once**,
and it marks which cells were newly written in that step, so you can watch the order slip fill in cell by cell.

### 🤔 Worksheet (write on paper first, then run and compare)
1. If the input is `print('hi'` (missing the closing parenthesis), which nodes does it pass through? What is `syntax_ok` at the end?
2. For the correct program, after which step does `feedback` first have content?
3. This system says "empty counts as not turned in". Does it take the write-feedback branch or the explain-error branch?

In [ ]:
#@title ▶ Run: watch the order slip fill in cell by cell (no AI yet)
homework = "Missing parenthesis"  #@param ["Missing parenthesis", "Correct program", "Empty", "Variable typo", "My homework"]
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import graph

helper = graph.build_helper()        # No model: give_feedback uses fixed text
code = load_homework(homework)
print(f"Homework: {code!r}\n")

steps, prev_state = [], None
for step_no, state in enumerate(helper.stream(graph.blank_order_slip(code), stream_mode="values"), start=1):
    print_order_slip(step_no, state, prev_state)
    steps.append(state)
    prev_state = state

print("\nFinal feedback:")
print(state["feedback"])
record(f"1-A Part 2: {homework} (no AI)", steps)

### 👀 What did you see?

1. **Missing parenthesis**: `check_syntax` writes `syntax_ok: False` and which line is wrong -> the conditional edge goes to `explain_error` -> it writes `feedback` -> end. It never passes through `give_feedback`.
2. **Correct program**: `syntax_ok: True` -> `give_feedback` writes `feedback` and `retries: 1` (step 3, the `give_feedback` step, is when the feedback first appears) -> `check_format` writes `format_ok: True` -> end.
3. **Empty**: empty is actually "legal" Python, but we added an extra `if` in `check_syntax`: empty counts as not turned in. This is "if something can be written as an if, write it in code". The output reuses the error branch's format with the title "Syntax", which does not mean empty is a syntax error.
- Only one or two cells change at each step: **each node writes only the cells it is responsible for**, and all other cells are passed along untouched.
- Whenever you read any LangGraph program from now on, ask two things first: which cells does State have? Which cell does each node write?

---
## Part 3: Put the AI into one node

> 📍 **Matching slides**: slide 26 "Putting AI into one node", 27 "The flow ended. Did the feedback pass?", slide 28 "Back to the helper's flow: all three design patterns"

In the first two parts, `give_feedback` only returned fixed text. In this part we swap it for a real AI and leave everything else alone.

Now `give_feedback` calls the model to write the feedback (the code is `make_give_feedback` in `graph.py`).
The AI is just one line of code living inside one node: **the flow diagram is unchanged, State is unchanged, and the arrows are unchanged**.

`check_format` is a plain-Python **checkpoint**: it checks the order of the "Strengths:" and "Improvements:" labels, that both sections are non-empty, and the character count; it does not judge whether the content is correct.
If the check fails and the limit has not been reached, it sends the work back to `give_feedback` to generate again. `retries` records how many times the feedback has been written; when it reaches "Maximum generations", it no longer sends it back and simply ends.

The form has two numbers you can adjust:

- **Maximum generations**: the most times the feedback is generated, counting the first one (set it to 3 and it generates at most 3 times).
  In the code this value is passed to `max_retries`, and the State field is called `retries`. Both are legacy names; what they actually count is "generations, including the first".
- **Feedback character limit**: the most characters the whole feedback may have (counting "Strengths:", "Improvements:", spaces, newlines and letters), and anything longer fails.

### 🤔 Prediction questions (write them down, then change the form and run)
First run once with the defaults (homework "Variable typo", maximum generations 3, feedback character limit 450) as a baseline, then change the form and compare:
1. Change "Maximum generations" to **1** and "Feedback character limit" to **90**: what will `format_ok` and the reason for stopping be?
2. The program ended, but `format_ok=False`. Has the format requirement been met?
3. (If you have time) Change generations back to 3 and keep the character limit at 90: will `retries` go up? If you run the same settings twice, are the feedback and `format_ok` the same? Record it honestly.

In [ ]:
#@title ▶ Run: AI writes the feedback + the program checks the format (regenerate if it fails)
homework = "Variable typo"  #@param ["Missing parenthesis", "Correct program", "Empty", "Variable typo", "My homework"]
max_generations = 3  #@param {type:"slider", min:1, max:5, step:1}
feedback_char_limit = 450  #@param {type:"slider", min:60, max:900, step:30}
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
import graph

helper = graph.build_helper(model, max_retries=max_generations, feedback_char_limit=feedback_char_limit)
code = load_homework(homework)

steps, prev_state = [], None
for step_no, state in enumerate(helper.stream(graph.blank_order_slip(code), stream_mode="values"), start=1):
    print_order_slip(step_no, state, prev_state)
    steps.append(state)
    prev_state = state

print(f"\nIn the end: the feedback was generated {state['retries']} time(s), format_ok = {state['format_ok']}")
if not state["syntax_ok"]:
    print("Reason for stopping: the homework did not pass the syntax/empty check, and the reason was explained.")
elif state["format_ok"]:
    print("Reason for stopping: the feedback passed the format check.")
else:
    print("Reason for stopping: reached the generation limit and the feedback still failed the format check, so a person needs to handle it. (Not passed.)")
if state.get("format_problem"):
    print(f"Reason the last attempt failed: {state['format_problem']}")
print("\nFinal feedback:")
print(state["feedback"])
record(f"1-A Part 3: {homework} (AI, generation limit {max_generations}, character limit {feedback_char_limit})", steps)

### 👀 What did you see?

- With a smaller character limit, the feedback may be sent back more often. Watch `retries`; it is not guaranteed to go up every time. When it reaches "Maximum generations" it stops, and `format_ok` may still be `False` then.
  **With no limit, it might loop forever**: how many times the feedback may be generated has to be limited by the program.
- Running the same homework twice, the feedback text may be the same or different. The feedback produced by `give_feedback` goes into the format check; the explanation for a syntax error takes a different branch. The flow **does not guarantee that the model will fix it until it passes**.
- Only `format_ok=True` means it passed the format check here. Hitting the generation limit also ends the flow, but the feedback may still fail. **The flow ending and the task being met are two different things.**
- Look back at the whole flow diagram: it happens to contain three common "flow layouts" (design patterns, from Anthropic's article "Building effective agents"):
  - **Prompt chaining**: `check_syntax -> give_feedback -> check_format`, small steps in a fixed order
  - **Routing**: the fork after `check_syntax`, which decides the type first and then hands off to a specialized step
  - **Evaluator-optimizer**: the loop where `check_format` sends work back to `give_feedback`; produce, check, and redo if it fails
- Counting characters: the limit counts every character of the feedback, including "Strengths:", "Improvements:", spaces and newlines. The count is by `len()`, so a long English sentence uses up the limit quickly.

---
## Optional after class: without "keep writing after the existing messages", the messages get overwritten

> 📍 **Matching slides**: slide 46 "Optional appendix: how messages in State are merged" (optional after class)

Every cell on the order slip so far held a single value (for example, after the check `syntax_ok` is True or False), so a new value can simply overwrite the old one.
But what if a cell needs to hold "a whole conversation"?

The model needs the program to supply the relevant conversation (1-0 Part 1). In LangGraph, the conversation goes in the `messages` cell of State.
If you don't tell LangGraph "this cell should keep appending" (using `add_messages`), by default it **overwrites the whole cell**, and the earlier conversation is gone.

### 🤔 Prediction questions
You send one message, "My name is Alex", through a node that calls the model. How many messages will `messages` contain afterwards? How many for each of the two ways of writing it?

In [ ]:
#@title ▶ Run: BadState (overwrite) vs. with add_messages (keep appending)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
from typing import Annotated, TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages

class BadState(TypedDict):
    messages: list                               # <- the default behavior is "overwrite"

class GoodState(TypedDict):
    messages: Annotated[list, add_messages]      # <- tells LangGraph: keep appending to this cell

def chatbot(state) -> dict:
    response = model.invoke(state["messages"])
    return {"messages": [response]}

for name, StateClass in [("BadState (no add_messages)", BadState), ("GoodState (with add_messages)", GoodState)]:
    builder = StateGraph(StateClass)
    builder.add_node("chatbot", chatbot)
    builder.add_edge(START, "chatbot")
    builder.add_edge("chatbot", END)
    result = builder.compile().invoke({"messages": [{"role": "user", "content": "My name is Alex"}]})
    print(f"{name}: messages has {len(result['messages'])} message(s)")
    for m in result["messages"]:
        print(f"   [{m.type if hasattr(m, 'type') else m['role']}] {(m.text if hasattr(m, 'text') else m['content'])[:60]}")
    record(f"1-A bonus: {name}", result)

### 👀 What did you see?

- `BadState` has only **1 message** left: the user's "My name is Alex" was overwritten by the model's reply.
- With `Annotated[list, add_messages]` there are **2 messages**: both the user's words and the model's reply are there.
- How to read that line: `messages` is a list, but when LangGraph handles it, it should merge rather than overwrite, using `add_messages`. A rule of thumb: **the first item in `Annotated` is the data type, and the rest is extra information for the framework.**
- The `MessagesState` in the optional after-class part of 1-B is exactly this line inside.

## Wrap-up: save your work and hand it in

- 💾 **Save to Google Drive**: save a copy of `/content/review-bot/` (your edited handbooks, code, and State records) to your own Google Drive.
- ⬇️ **Download the zip**: download it straight to your computer.
- 📸 **The last cell**: prints the results the experiment cells saved. Pick the relevant records for a screenshot, then add your observations and reasons as the assignment asks.

These are the results the experiment cells saved through `record()`, not every intermediate state of every run.
Records in the same runtime pile up, so they may include experiments from other notebooks; use the printed titles to tell them apart.

A temporary browser disconnect does not necessarily wipe your files; files under `/content/` are lost only when the runtime is deleted or recycled.
To keep your handbooks, code, and checkpoint, back them up first. The notebook itself is saved with the menu "File → Save a copy in Drive". The two kinds of saving do different jobs.
When you come back, first check that the runtime is still there. If it was reset, re-run "0. Setup" and this part's setup cell, then restore the files you backed up as needed.

In [ ]:
#@title 💾 Save to Google Drive (an authorization window will pop up)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
save_to_drive()

In [ ]:
#@title ⬇️ Download the zip
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
download_zip()

In [ ]:
#@title 📸 Print the recorded experiment results (pick the relevant records to hand in)
import sys, importlib; sys.path.insert(0, "/content/review-bot"); importlib.invalidate_caches()
try:
    from workshop import *   # prepare, record, ... (the helpers written by "0. Setup")
except ImportError:
    raise SystemExit("⚠️ Environment not found: run the 0. Setup cell at the very top first")
model = prepare()
print_history()